# OmniEVIE-4B — multimodal multi-vector retrieval on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/OmniEVIE-4B-multimodal-retrieval.ipynb)

[`tencent/OmniEVIE-4B`](https://huggingface.co/tencent/OmniEVIE-4B) is Tencent's **4.54B** unified multi-modal, multi-vector retriever (Apache-2.0, public), fine-tuned from [`Qwen/Qwen3.5-4B`](https://huggingface.co/Qwen/Qwen3.5-4B). The card lists it for **text, natural images, visual documents, native video**, and agent / tool / GUI / memory retrieval. It emits **one vector per token** (ColBERT-style MaxSim), not a single pooled embedding. Hub weights are one **9.1 GB** `model.safetensors` shard (bf16) plus a tiny `query_gate.safetensors`. Prefix-MRL dimensions at runtime are `{64, 128, 256, 512, 1024, 2048}`.

**Why this model.** One Apache-2.0 checkpoint that ranks text against text *and* against page/figure images, using the card's Sentence Transformers `MultiVectorEncoder` path. That is the drop-in prototyping API on the card. The recommended production package (`omnievie` + FlashAttention-2) is **not** used here: a T4 cannot run FlashAttention-2.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/OmniEVIE-4B-multimodal-retrieval.md) for when this notebook is a good fit and when it is not.

**Licence:** Apache-2.0. Weights are public — `HF_TOKEN` is optional.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has about 15 GB of VRAM, **no native bfloat16**, and **no FlashAttention-2**. The checkpoint dtype is bfloat16; this notebook loads **float16 + SDPA** (eager fallback). Hub `model.safetensors` is **9.1 GB**. Measured peak on a Colab T4 is **8.7 GB** (`torch.cuda.max_memory_allocated`) with `batch_size=1` and images capped on the long edge. Scores will not match the card's printed bf16 decimals.

**Not in this notebook:** the `omnievie` CLI / `ThreeStageIndex` / `QueryGate` / page-box localization path (needs `flash-attn>=2.7.0`), native video, million-scale CPU search, or a rerun of MMEB / ViDoRe numbers.

**Links:** [model card](https://huggingface.co/tencent/OmniEVIE-4B) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/OmniEVIE-4B-multimodal-retrieval.md) · [GitHub](https://github.com/Tencent/OmniEVIE) · [report](https://arxiv.org/abs/2611.32203) · [announcement](https://x.com/paperswithcodee/status/2125530235802100031)


## 1. Check the GPU

Print `torch.cuda.is_available()`, the device name, and free VRAM. This cell imports **torch only** so a later transformers / sentence-transformers pin can take effect without a restart. T4 (`sm_75`) has no native bfloat16 — **float16** is the path this notebook uses.


In [ ]:
import os

import torch

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8

# Checkpoint dtype is bfloat16. T4 is Turing sm_75 — stay on float16.
DTYPE = torch.float16
free_b, total_b = torch.cuda.mem_get_info()
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {total_b / 1024**3:.2f} GB | free: {free_b / 1024**3:.2f} GB")
print("native bf16:", NATIVE_BF16, "| FlashAttention-2: off (sdpa / eager)")
if major < 8:
    print("No native bfloat16 on this GPU; loading float16 (~9.1 GB weights).")


## 2. Install Sentence Transformers

The [card](https://huggingface.co/tencent/OmniEVIE-4B) requires `sentence-transformers[image,video]>=6.3.0` and `transformers>=5.13.1`, and loads `MultiVectorEncoder` with **no** `trust_remote_code`. This notebook installs the **`[image]`** extra (PIL / URL documents) and **does not** install the video extra or `flash-attn`. Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled `torch`. Keep Colab's torch. The install cell then drops any already-imported `transformers` / `sentence_transformers` modules so **Runtime > Run all** can pick up the pin without *Restart session*. If the version assert still fails, restart once and continue from the GPU cell.


In [ ]:
%pip install -q -U "sentence-transformers[image]>=6.3.0" "transformers>=5.13.1" pillow

import sys

# Card: sentence-transformers[image,video]>=6.3.0, transformers>=5.13.1,
# MultiVectorEncoder, trust_remote_code not required.
# Video extra is skipped (no video demo on a free T4).
# Do NOT install flash-attn (T4 cannot run FlashAttention-2).
# Do NOT replace Colab's CUDA torch wheel.

for name in list(sys.modules):
    if name == "transformers" or name.startswith("transformers."):
        del sys.modules[name]
    if name == "sentence_transformers" or name.startswith("sentence_transformers."):
        del sys.modules[name]

import sentence_transformers
import transformers
from packaging.version import Version

assert Version(sentence_transformers.__version__.split("+")[0]) >= Version("6.3.0"), (
    f"sentence-transformers {sentence_transformers.__version__} is loaded; OmniEVIE-4B needs "
    "sentence-transformers>=6.3.0. Runtime > Restart session, then rerun from the GPU check cell."
)
assert Version(transformers.__version__.split("+")[0]) >= Version("5.13.1"), (
    f"transformers {transformers.__version__} is loaded; OmniEVIE-4B needs "
    "transformers>=5.13.1. Runtime > Restart session, then rerun from the GPU check cell."
)
print("sentence-transformers", sentence_transformers.__version__)
print("transformers", transformers.__version__)


## 3. Load OmniEVIE-4B

`MultiVectorEncoder` is the class the [card's Sentence Transformers snippet](https://huggingface.co/tencent/OmniEVIE-4B) uses. `model_kwargs` is forwarded to the Hugging Face tower so the T4 load is **float16 + SDPA** rather than the checkpoint's bfloat16 default. First run downloads about **9.1 GB**. Attention is **sdpa**, then **eager**. The card's tip: `batch_size=1` so left-padding does not perturb Qwen3.5 linear-attention layers.

This is **not** `OmniEVIE.from_pretrained` from the `omnievie` package. That path sets `attn_implementation="flash_attention_2"` and needs `flash-attn>=2.7.0`.


In [ ]:
import gc

from sentence_transformers import MultiVectorEncoder

MODEL_ID = "tencent/OmniEVIE-4B"
BATCH_SIZE = 1  # card: batch_size=1 yields exact reference embeddings


def print_vram(label=""):
    suffix = f" ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM{suffix}: allocated {allocated:.2f} GB | reserved {reserved:.2f} GB | "
        f"peak {peak:.2f} GB | free {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
    )


def load_encoder(attn):
    return MultiVectorEncoder(
        MODEL_ID,
        device="cuda",
        model_kwargs={"dtype": DTYPE, "attn_implementation": attn},
    )


torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

ATTN = None
model = None
last_error = None
for attn in ("sdpa", "eager"):
    try:
        model = load_encoder(attn)
        ATTN = attn
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} float16 load failed:", exc)
        del model
        model = None
        gc.collect()
        torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("could not load tencent/OmniEVIE-4B in float16") from last_error

n_params = sum(p.numel() for p in model.parameters()) / 1e9
print("attention:", ATTN, "| dtype:", next(model.parameters()).dtype)
print(f"parameters: {n_params:.2f}B")
print("prompts:", getattr(model, "prompts", None))
print("similarity:", getattr(model, "similarity_fn_name", None))
print_vram("after load")


## 4. Demo 1 — text-to-text retrieval

Card **Text Retrieval** snippet: `encode_query` / `encode_document` / `similarity` (MaxSim). Two queries, three sentences. `batch_size=1`. Printed scores on the card were bf16; this T4 run is float16, so the decimals will differ. Ranked rows should still put each query on its matching sentence.


In [ ]:
queries = [
    "Two TV show hosts sit on a red couch and discuss the news.",
    "Did the FDA's 2018 Strategic Policy Roadmap eliminate clinical trials?",
]
documents = [
    "Two television hosts discuss the news while sitting on a red couch.",
    "The FDA's 2018 Strategic Policy Roadmap proposed modernizing clinical trials, not eliminating them.",
    "A dog plays with a ball in a park.",
]

query_embeddings = model.encode_query(queries, batch_size=BATCH_SIZE)
document_embeddings = model.encode_document(documents, batch_size=BATCH_SIZE)
print("query shapes:", [tuple(embedding.shape) for embedding in query_embeddings])
print("document shapes:", [tuple(embedding.shape) for embedding in document_embeddings])

# Card Prefix-MRL: truncate_dim={64, 128, 256, 512, 1024} (full default is 2048).
short = model.encode_query(queries[:1], batch_size=BATCH_SIZE, truncate_dim=64)
print("truncate_dim=64 query shape:", tuple(short[0].shape))

scores = model.similarity(query_embeddings, document_embeddings)
if not isinstance(scores, torch.Tensor):
    scores = torch.tensor(scores)
scores = scores.detach().float().cpu()
print("MaxSim scores:\n", scores)

for row, query in enumerate(queries):
    print(f"\nQ: {query}")
    order = torch.argsort(scores[row], descending=True).tolist()
    for rank, col in enumerate(order, start=1):
        print(f"  {rank}. {scores[row, col].item():.4f}  {documents[col]}")

print_vram("after text retrieval")


## 5. Demo 2 — text-to-image retrieval

Card **Text-to-Image Retrieval** snippet. Documents are public figure URLs from the [Hub repo](https://huggingface.co/tencent/OmniEVIE-4B/tree/main/assets) (the same files the card uses). Images are downloaded and capped at **768 px** on the long edge before `encode_document` (measured Colab T4 peak **8.7 GB**); the processor default allows up to `max_pixels=1,048,576`. Query is the card's: *"Which model ranks first on the MMEB leaderboard?"*


In [ ]:
from io import BytesIO

import requests
from PIL import Image

ASSET = "https://huggingface.co/tencent/OmniEVIE-4B/resolve/main/assets"
IMAGE_DOCS = [
    ("fig1_overview.png", f"{ASSET}/fig1_overview.png"),
    ("fig4_sthac.png", f"{ASSET}/fig4_sthac.png"),
    ("mmeb_leaderboard.png", f"{ASSET}/mmeb_leaderboard.png"),
]
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}
LONGEST = 768  # T4 cap; card processor longest_edge is 1_048_576 pixels of area


def load_image(url, longest=LONGEST):
    response = requests.get(url, headers=UA, timeout=120)
    response.raise_for_status()
    image = Image.open(BytesIO(response.content)).convert("RGB")
    image.thumbnail((longest, longest))
    return image


image_inputs = []
for name, url in IMAGE_DOCS:
    image = load_image(url)
    image_inputs.append(image)
    print(f"{name}: {image.size}")

image_query = ["Which model ranks first on the MMEB leaderboard?"]
image_q_emb = model.encode_query(image_query, batch_size=BATCH_SIZE)
image_d_emb = model.encode_document(image_inputs, batch_size=BATCH_SIZE)
image_scores = model.similarity(image_q_emb, image_d_emb)
if not isinstance(image_scores, torch.Tensor):
    image_scores = torch.tensor(image_scores)
image_scores = image_scores.detach().float().cpu()
print("MaxSim scores:\n", image_scores)

print(f"\nQ: {image_query[0]}")
order = torch.argsort(image_scores[0], descending=True).tolist()
for rank, col in enumerate(order, start=1):
    print(f"  {rank}. {image_scores[0, col].item():.4f}  {IMAGE_DOCS[col][0]}")
print("top image:", IMAGE_DOCS[int(image_scores[0].argmax())][1])
print_vram("after text-to-image")


## 6. Demo 3 — visual-document page query

The card treats visual documents as images passed to `encode_document` (PIL, path, or URL; no OCR step). This cell ranks three more Hub figures that look like paper pages / diagrams: ST-HAC compression, the three-stage retrieval funnel, and token-level alignment. Query: *"Where is the three-stage retrieval funnel latency breakdown?"*


In [ ]:
PAGE_DOCS = [
    ("fig4_sthac.png", f"{ASSET}/fig4_sthac.png"),
    ("fig5_three_stage.png", f"{ASSET}/fig5_three_stage.png"),
    ("fig7_alignment.png", f"{ASSET}/fig7_alignment.png"),
]

page_inputs = []
for name, url in PAGE_DOCS:
    image = load_image(url)
    page_inputs.append(image)
    print(f"{name}: {image.size}")

page_query = ["Where is the three-stage retrieval funnel latency breakdown?"]
page_q_emb = model.encode_query(page_query, batch_size=BATCH_SIZE)
page_d_emb = model.encode_document(page_inputs, batch_size=BATCH_SIZE)
page_scores = model.similarity(page_q_emb, page_d_emb)
if not isinstance(page_scores, torch.Tensor):
    page_scores = torch.tensor(page_scores)
page_scores = page_scores.detach().float().cpu()
print("MaxSim scores:\n", page_scores)

print(f"\nQ: {page_query[0]}")
order = torch.argsort(page_scores[0], descending=True).tolist()
for rank, col in enumerate(order, start=1):
    print(f"  {rank}. {page_scores[0, col].item():.4f}  {PAGE_DOCS[col][0]}")
print("top page:", PAGE_DOCS[int(page_scores[0].argmax())][1])
print_vram("after visual-document query")


## 7. Caveats and cleanup

- **Sentence Transformers, not `omnievie`.** The card's recommended production runtime (`pip install git+https://github.com/Tencent/OmniEVIE.git`) requires `flash-attn>=2.7.0` and `attn_implementation="flash_attention_2"`. A T4 cannot run FlashAttention-2. This notebook uses the card's **Sentence Transformers** path (`MultiVectorEncoder`, uncompressed token embeddings, MaxSim).
- **Float16 on a T4.** Saved dtype is bfloat16. T4 has no native bfloat16. No FlashAttention-2. Printed card scores (for example `23.0312`) are bf16 reference values and will not match.
- **`batch_size=1`.** The card: this avoids left-padding tokens perturbing Qwen3.5 linear-attention layers.
- **VRAM.** Hub `model.safetensors` is **9.1 GB** bf16. Measured peak on a Colab T4 is **8.7 GB** (`torch.cuda.max_memory_allocated`) with `batch_size=1` and 768 px images. Video is not run here.
- **Prefix-MRL.** `truncate_dim` of 64 / 128 / 256 / 512 / 1024 is a card option. Demo 1 prints a 64-d prefix; ranking uses the full 2048-d vectors.
- **Query gate / ST-HAC / three-stage index.** `query_gate.safetensors`, int8 `K128×d64` compression, and million-scale CPU search belong to the `omnievie` package, not this Sentence Transformers export.
- **Published scores are not rerun.** MMEB-Leaderboard / MMEB-V2 / ViDoRe V3 / latency tables on the card are the authors' numbers.
- **Apache-2.0.** Public weights; cancel Colab `HF_TOKEN` prompts. This notebook sets `HF_HUB_DISABLE_IMPLICIT_TOKEN=1`.

The next cell drops the model so you can load something else on the same T4. It prints `torch.cuda.max_memory_allocated` for this session.


In [ ]:
print_vram("before cleanup")
print(f"torch.cuda.max_memory_allocated: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
try:
    del model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print_vram("after cleanup")
    print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("OmniEVIE-4B released. Re-run the load cell if you want another query.")
